
# Dear Canada — Knowledge Graph Extraction Pipeline (v3)

**v3 changes from v2**, per review of the first SofS run:

- `SignificantEvent` node/registry **removed entirely** — it was too coarse
  to be useful and showed up on too many entries. Historical-event linkage
  is deferred to a later, corpus-wide pass that joins on date + location +
  `condition_name`/`incident_type`, rather than being forced at extraction
  time.
- `travels_to` / `located_in` **replaced by a reified `Stay` node.** A plain
  edge can't itself be the target of another edge, so there was nowhere to
  attach a `Date` node to "the event of being somewhere" — only to a
  property on the edge. Reifying presence/movement as its own node fixes
  this and, as a side effect, gives you one stop = one node, so "the many,
  many stops" on a migration are just a chain of `Stay` nodes.
- `impacted_by` **replaced by `HealthState`** (incur/recover/die, with
  `mechanism` + a literal, specific `condition_name` — "typhus", not
  "illness") **and by `Incident`** (for discrimination: victim, optional
  perpetrator, optional grounds).
- **`grieves_for` removed entirely**, no replacement.
- `Location` and `Organization` are now split: `Location` is the physical
  place, `Organization` is the institution, linked by `located_at`.
- `Date` is now a real node (not just an entry property), reached via
  `OCCURS_ON` edges from `Entry`, `Stay`, `HealthState`, and `Incident`.
- `Entry` nodes now explicitly carry `entry_id` as a named property.
- Output is restructured: one JSON file per node type, plus a single flat
  `edges.json` (discriminated by `edge_type`) rather than a bespoke file per
  relation category — this scales better now that several node types
  (`Stay`, `HealthState`, `Incident`) each generate multiple edges.
- **Efficiency**: the static system prompt + book synopsis are now cached
  (`cache_control: ephemeral`) instead of resent every call, and the state
  summary is capped to recently-touched entities instead of growing for the
  whole book — see §1 and §7.
- Still **one entry per LLM call** (unchanged, per your instruction).

## Final v3 schema (as agreed)

**Nodes**: `Character` (+`historicity`), `Location`, `Region`,
`Organization`, `IdentityCategory` (literal, unmerged), `Entry`
(+`entry_id`), `Date`, `Stay`, `HealthState`, `Incident`.

**Edges**: `works_for`, `works_with`, `student_of`, `studies_with`,
`lives_with`, `stops_living_with`, `is_related_to` (+`kinship_label`,
literal), `social_relationship` (+`valence`), `within` (Location/Region →
Region, one direction only — `contains` dropped as a redundant inverse),
`located_at` (Organization → Location), `has_identity_descriptor`
(+`raw_text`, `attributed_by`), `has_stay` / `at` / `occurs_on` /
`recorded_in` (Stay's edges), `incurs` / `recovers_from` / `dies_from` /
`perpetrated_by` (HealthState's edges), `against` / `perpetrated_by` /
`on_grounds_of` (Incident's edges).


## 0. Setup & configuration

In [11]:

import os
import re
import json
import glob
import time
from dataclasses import dataclass, field, asdict
from pathlib import Path
import anthropic
import pandas as pd
from tqdm.auto import tqdm

# --- paths -------------------------------------------------------------
# Versioned output directory: v3's schema is NOT compatible with v2's
# output (different node/edge shapes entirely), so this writes to its own
# folder rather than overwriting or mixing with v2 files. See the "Running
# this on A Sea of Sorrows" section at the end for what to do with old files.
DATA_DIR = Path("./data")
OUT_DIR = Path("./output/v3")
REGISTRY_DIR = OUT_DIR / "registries"
CANDIDATE_DIR = Path("./output/candidate_lists")   # shared across versions, see §4
REVIEW_DIR = OUT_DIR / "for_review"

for d in (OUT_DIR, REGISTRY_DIR, CANDIDATE_DIR, REVIEW_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- pilot scope ---------------------------------------------------------
BOOK_CODE = "SofS_entries_halved"   # A Sea of Sorrows (Norah McClintock) — pilot book

In [3]:
from dotenv import load_dotenv

env_path = Path.cwd() / ".env"
print(f"Looking for: {env_path}")
print(f"Found: {env_path.exists()}")

load_dotenv(env_path, override=True)

print("Key loaded:", bool(os.getenv("ANTHROPIC_API_KEY")))

Looking for: /Users/sophiebird/Downloads/Dear Canada/NLP and Data Creation from Anthology/.env
Found: True
Key loaded: True


In [4]:
# --- LLM config ------------------------------------------------------------
# Set your key as an environment variable before running:
#   export ANTHROPIC_API_KEY=sk-ant-...
client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

# Check https://docs.claude.com/en/docs/about-claude/models/overview for the
# current model lineup before running at full-corpus scale.
MODEL_NAME = "claude-sonnet-5"

# How many prior entries (full text) to include as context for the sliding
# window, in addition to the compact state_summary. This is for resolving
# pronouns/references (as in the textbook's "R." / "Rossby" example), not for
# inferring continuity — continuity (location carry-forward, etc.) is
# handled downstream in code.
CONTEXT_WINDOW_ENTRIES = 2
STATE_SUMMARY_RECENT_WINDOW = 15   # entries; entities not touched within this
                                    # window get collapsed to bare id+label
                                    # in the state summary (see §7)


## 1. Load book CSV

Unchanged from v2. Expects `book, chapter_id, entry_id, date, entry_text`.


In [5]:

def load_book(book_code: str, data_dir: Path = DATA_DIR) -> pd.DataFrame:
    path = data_dir / f"{book_code}.csv"
    df = pd.read_csv(path)
    required_cols = {"book", "chapter_id", "entry_id", "date", "entry_text"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"{path} is missing columns: {missing}")
    df["_chapter_num"] = df["chapter_id"].str.extract(r"(\d+)").astype(int)
    df["_entry_num"] = df["entry_id"].str.extract(r"e(\d+)$").astype(int)
    df = df.sort_values(["_chapter_num", "_entry_num"]).reset_index(drop=True)
    return df

book_df = load_book(BOOK_CODE)



## 2. NER recall net

Unchanged mechanically from v2 — spaCy still lumps all place mentions
(GPE/LOC/FAC) into one `candidate_locations` bucket. The **split into
`Location` vs `Region` now happens in the review step (§4)**, not here,
since spaCy's own categories don't line up with that distinction (a "FAC"
tag doesn't tell you whether something is a farm vs. a province).


In [7]:

import spacy

try:
    nlp = spacy.load("en_core_web_trf")
except OSError:
    nlp = spacy.load("en_core_web_sm")

PERSON_LABELS = {"PERSON"}
PLACE_LABELS = {"GPE", "LOC", "FAC"}
ORG_LABELS = {"ORG"}

def ner_pass_book(df: pd.DataFrame) -> dict:
    names, places, orgs = {}, {}, {}
    for _, row in df.iterrows():
        doc = nlp(row["entry_text"])
        for ent in doc.ents:
            if ent.label_ in PERSON_LABELS:
                target = names
            elif ent.label_ in PLACE_LABELS:
                target = places
            elif ent.label_ in ORG_LABELS:
                target = orgs
            else:
                continue
            key = ent.text.strip()
            if not key:
                continue
            if key not in target:
                target[key] = {"count": 0, "example_entry_id": row["entry_id"], "label": ent.label_}
            target[key]["count"] += 1
    return {"candidate_characters": names, "candidate_places": places, "candidate_organizations": orgs}

def save_candidate_lists(book_code: str, candidates: dict):
    path = CANDIDATE_DIR / f"{book_code}_candidates.json"
    with open(path, "w") as f:
        json.dump(candidates, f, indent=2)
    print(f"Wrote {path} — {len(candidates['candidate_characters'])} character forms, "
          f"{len(candidates['candidate_places'])} place forms (locations+regions, unsplit), "
          f"{len(candidates['candidate_organizations'])} organization forms.")
    return path

candidates = ner_pass_book(book_df)
save_candidate_lists(BOOK_CODE, candidates)


/Users/sophiebird/anaconda3/envs/dearcanadaenv/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Wrote output/candidate_lists/SofS_entries_halved_candidates.json — 77 character forms, 29 place forms (locations+regions, unsplit), 3 organization forms.


PosixPath('output/candidate_lists/SofS_entries_halved_candidates.json')


## 3. Review candidate lists

Updated from v2: the canonicalization prompt now (a) splits place mentions
into `locations` (physical: buildings, farms, ships, named stops) vs.
`regions` (countries, provinces, cities, "the ocean"), (b) asks for an
`org_type_hint` on organizations, and (c) asks for a `historicity_hint` per
character (fictional / historical / ambiguous) — all three are **hints for
you to confirm or correct**, not final values; review the draft before
treating it as ground truth, exactly as before.

**If you have a v2 `SofS_reviewed.json`**: the character/location/org
identities themselves haven't changed, only the schema around them. It's
probably faster to re-run `suggest_canonical_entities` fresh (it's cheap,
one call) than to hand-patch the old file with the three new fields plus
splitting its `locations` list into `locations`+`regions`.


In [13]:

CANONICALIZE_SYSTEM_PROMPT = """You are cleaning up raw named-entity lists \
extracted from one book of a children's historical fiction diary novel \
(the Dear Canada series). You will be given candidate CHARACTER surface \
forms, candidate PLACE surface forms (locations and regions mixed \
together), and candidate ORGANIZATION surface forms, each with an \
occurrence count.

Your job:
1. Group surface forms that refer to the same real-world entity.
2. Split the PLACE candidates into two separate lists:
   - "locations": physical, specific places — buildings, farms, ships, \
named migration stops, a specific quarantine station, a specific house.
   - "regions": larger administrative/geographic areas — countries, \
provinces, cities, "the ocean", "the sea".
3. For each character, suggest a historicity: "fictional" (an \
invented character), "historical" (a real documented person), or \
"ambiguous" (unclear / plausibly either) Most characters in this corpus are fictional, but some are based on real people.\
4. For each organization, suggest an org_type (e.g. "religious \
order", "hospital", "school", "charity", "government body") or null if \
unclear.
5. Flag likely false positives (ships/vehicles mistagged as \
organizations, the narrator referring to herself, common nouns mistagged \
as places, etc).

Do NOT canonicalize or group anything related to identity, ethnicity, \
religion, or nationality (e.g. do not create a "the Irish" entity) — that \
is handled separately and kept as literal text elsewhere in this pipeline.

Return ONLY valid JSON, no preamble, in this shape:
{
  "characters": [
    {"suggested_id": "johanna_leary", "canonical_label": "Johanna Leary", "aliases": ["Johanna", "I", "me"], "relation_to_narrator": "self", "historicity": "fictional"}
  ],
  "locations": [
    {"suggested_id": "grosse_ile_fever_sheds", "canonical_label": "the Grosse Île fever sheds", "aliases": ["the fever sheds", "the quarantine sheds"]}
  ],
  "regions": [
    {"suggested_id": "canada_east", "canonical_label": "Canada East", "aliases": ["Canada East", "Quebec"]}
  ],
  "organizations": [
    {"suggested_id": "...", "canonical_label": "...", "aliases": ["..."], "org_type": "e.g. religious order / hospital / school, or null"}
  ],
  "likely_false_positives": ["..."]
}
"""

def suggest_canonical_entities(candidates: dict) -> dict:
    user_content = json.dumps({
        "candidate_characters": {k: v["count"] for k, v in candidates["candidate_characters"].items()},
        "candidate_places": {k: v["count"] for k, v in candidates["candidate_places"].items()},
        "candidate_organizations": {k: v["count"] for k, v in candidates["candidate_organizations"].items()},
    }, indent=2)

    with client.messages.stream(
        model=MODEL_NAME,
        max_tokens=30000,
        system=CANONICALIZE_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_content}],
    ) as stream:
        chunks = []

        with tqdm(
            desc="Claude response",
            unit=" chars",
            unit_scale=True,
        ) as progress:
            for chunk in stream.text_stream:
                chunks.append(chunk)
                progress.update(len(chunk))

        resp = stream.get_final_message()

    text = "".join(chunks).strip()

    if resp.stop_reason == "max_tokens":
        raise RuntimeError(
            "Claude used the entire token limit before producing complete JSON."
        )

    if not text:
        raise RuntimeError(
            f"Claude returned no text. stop_reason={resp.stop_reason!r}; "
            f"content={resp.content!r}"
        )

    print(f"Received {len(text):,} characters. First 300:\n{text[:300]!r}")
    return parse_claude_json(text)

def save_reviewed_suggestion(book_code: str, suggestion: dict):
    path = CANDIDATE_DIR / f"{book_code}_reviewed_DRAFT.json"
    with open(path, "w") as f:
        json.dump(suggestion, f, indent=2)
    print(f"Wrote DRAFT to {path} — review and rename to *_reviewed.json before use.")
    return path

def load_reviewed_entities(book_code: str) -> dict:
    path = CANDIDATE_DIR / f"{book_code}_reviewed.json"
    with open(path) as f:
        return json.load(f)

def parse_claude_json(text: str) -> dict:
    text = text.strip()

    # Remove a Markdown code fence if Claude included one.
    if text.startswith("```"):
        lines = text.splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        text = "\n".join(lines).strip()

    # If there is a short preamble before the JSON, start at the JSON object.
    start = text.find("{")
    if start == -1:
        raise RuntimeError(
            "Claude did not return a JSON object. "
            f"Response begins: {text[:500]!r}"
        )

    decoder = json.JSONDecoder()
    try:
        value, end = decoder.raw_decode(text[start:])
    except json.JSONDecodeError as exc:
        raise RuntimeError(
            f"Claude returned invalid JSON: {exc}. "
            f"Response begins: {text[:500]!r}"
        ) from exc

    if not isinstance(value, dict):
        raise RuntimeError(
            f"Expected a JSON object, got {type(value).__name__}."
        )

    return value


suggestion = suggest_canonical_entities(candidates)
save_reviewed_suggestion(BOOK_CODE, suggestion)


Claude response: 0.00 chars [00:00, ? chars/s]

Received 12,215 characters. First 300:
'```json\n{\n  "characters": [\n    {\n      "suggested_id": "johanna_leary",\n      "canonical_label": "Johanna Leary",\n      "aliases": ["Johanna", "Johanna Leary", "The Johanna Leary", "a Joanna Leary", "Leary", "I", "me"],\n      "relation_to_narrator": "self",\n      "historicity": "fictional"\n    },\n '
Wrote DRAFT to output/candidate_lists/SofS_entries_halved_reviewed_DRAFT.json — review and rename to *_reviewed.json before use.


PosixPath('output/candidate_lists/SofS_entries_halved_reviewed_DRAFT.json')

In [16]:
# (hand-review the _DRAFT file, save as *_reviewed.json, then:)
reviewed = load_reviewed_entities(BOOK_CODE)


## 4. Registries

`CharacterRegistry`, `LocationRegistry`, `RegionRegistry`,
`OrganizationRegistry` follow the same alias-resolution pattern as v2
(within-book canonical identity, auto-register new entities with a
`pending_review` flag). Two new, simpler registries:

- **`DateRegistry`**: fully deterministic, no LLM involvement — it just
  turns date strings the model already extracted into `Date` nodes with an
  inferred `precision` (day/month/year/unknown). This is the mechanism that
  finally lets a `Date` have anything pointing at it.
- **`IdentityCategoryRegistry`**: matches on **exact literal text only**
  (case/whitespace-normalized), no fuzzy aliasing, no semantic merging. So
  "the Irish girl" used twice collapses to one node (it's clearly the same
  phrase), but "the Irish girl" and "a Papist" stay separate nodes — which
  is exactly the literal-capture, no-normalization behavior you asked for.


In [17]:

@dataclass
class CharacterEntry:
    character_id: str
    canonical_label: str
    aliases: list = field(default_factory=list)
    relation_to_narrator: str = "unknown"
    historicity: str = "unknown"   # fictional | historical | ambiguous | unknown
    last_known_location: str = None
    last_location_stated_entry: str = None
    first_mentioned_entry: str = None
    last_touched_index: int = -1

@dataclass
class LocationEntry:
    location_id: str
    canonical_label: str
    aliases: list = field(default_factory=list)
    first_mentioned_entry: str = None
    last_touched_index: int = -1
    coordinates: list = None   # filled in a later, separate geocoding pass

@dataclass
class RegionEntry:
    region_id: str
    canonical_label: str
    aliases: list = field(default_factory=list)
    first_mentioned_entry: str = None
    last_touched_index: int = -1

@dataclass
class OrganizationEntry:
    org_id: str
    canonical_label: str
    aliases: list = field(default_factory=list)
    org_type_hint: str = None
    first_mentioned_entry: str = None
    last_touched_index: int = -1


class _AliasRegistryBase:
    entry_cls = None
    id_field = None

    def __init__(self):
        self._by_id = {}
        self._alias_to_id = {}
        self.pending_review = []

    def seed_from_reviewed(self, reviewed_list, extra_fields=None):
        extra_fields = extra_fields or {}
        for item in reviewed_list:
            kwargs = dict(canonical_label=item["canonical_label"], aliases=list(item.get("aliases", [])))
            kwargs[self.id_field] = item["suggested_id"]
            for k, source_key in extra_fields.items():
                kwargs[k] = item.get(source_key)
            entry = self.entry_cls(**kwargs)
            self._by_id[getattr(entry, self.id_field)] = entry
            for a in entry.aliases + [entry.canonical_label]:
                self._alias_to_id[a.lower()] = getattr(entry, self.id_field)

    def resolve(self, label: str):
        if not label:
            return None
        return self._by_id.get(self._alias_to_id.get(label.lower()))

    def canonical_label(self, label: str) -> str:
        e = self.resolve(label)
        return e.canonical_label if e else label

    def register_or_flag(self, label: str, entry_id: str, **extra_kwargs):
        existing = self.resolve(label)
        if existing:
            return existing
        new_id = re.sub(r"[^a-z0-9]+", "_", label.lower()).strip("_")
        kwargs = dict(canonical_label=label, aliases=[label], first_mentioned_entry=entry_id, **extra_kwargs)
        kwargs[self.id_field] = new_id
        entry = self.entry_cls(**kwargs)
        self._by_id[new_id] = entry
        self._alias_to_id[label.lower()] = new_id
        self.pending_review.append({self.id_field: new_id, "label": label, "entry_id": entry_id})
        return entry

    def touch(self, entity_id: str, index: int):
        if entity_id in self._by_id:
            self._by_id[entity_id].last_touched_index = index

    def export(self) -> list:
        return [asdict(v) for v in self._by_id.values()]


class CharacterRegistry(_AliasRegistryBase):
    entry_cls = CharacterEntry
    id_field = "character_id"

    def seed_from_reviewed(self, reviewed_characters: list):
        super().seed_from_reviewed(reviewed_characters, extra_fields={
            "relation_to_narrator": "relation_to_narrator",
            "historicity": "historicity_hint",
        })
        for e in self._by_id.values():
            e.relation_to_narrator = e.relation_to_narrator or "unknown"
            e.historicity = e.historicity or "unknown"

    def seed_narrator(self, narrator_id: str, narrator_label: str, aliases=None, historicity: str = "fictional"):
        if narrator_id in self._by_id:
            return self._by_id[narrator_id]
        entry = CharacterEntry(character_id=narrator_id, canonical_label=narrator_label,
                                aliases=(aliases or []) + ["I", "me", "my", "myself"],
                                relation_to_narrator="self", historicity=historicity)
        self._by_id[narrator_id] = entry
        for a in entry.aliases + [entry.canonical_label]:
            self._alias_to_id[a.lower()] = narrator_id
        return entry

    def register_or_flag(self, label: str, entry_id: str, relation_hint: str = "unknown"):
        return super().register_or_flag(label, entry_id, relation_to_narrator=relation_hint, historicity="unknown")

    def update_location(self, character_id: str, location_label: str, entry_id: str):
        if character_id in self._by_id and location_label:
            self._by_id[character_id].last_known_location = location_label
            self._by_id[character_id].last_location_stated_entry = entry_id


class LocationRegistry(_AliasRegistryBase):
    entry_cls = LocationEntry
    id_field = "location_id"


class RegionRegistry(_AliasRegistryBase):
    entry_cls = RegionEntry
    id_field = "region_id"


class OrganizationRegistry(_AliasRegistryBase):
    entry_cls = OrganizationEntry
    id_field = "org_id"

    def seed_from_reviewed(self, reviewed_organizations: list):
        super().seed_from_reviewed(reviewed_organizations, extra_fields={"org_type_hint": "org_type_hint"})


def infer_date_precision(date_str: str) -> str:
    if not date_str:
        return "unknown"
    parts = str(date_str).split("-")
    return {3: "day", 2: "month", 1: "year"}.get(len(parts), "unknown")


class DateRegistry:
    """Deterministic, code-only — never asks the LLM to create or resolve
    a Date node, only to supply date strings. One node per distinct
    normalized-date string (full or partial)."""

    def __init__(self):
        self._by_str = {}

    def get_or_create(self, date_str: str):
        if not date_str:
            return None
        if date_str not in self._by_str:
            self._by_str[date_str] = {
                "date_id": f"date_{date_str}",
                "normalized_date": date_str,
                "precision": infer_date_precision(date_str),
            }
        return self._by_str[date_str]["date_id"]

    def export(self) -> list:
        return list(self._by_str.values())


class IdentityCategoryRegistry:
    """Exact-literal-text matching only — no fuzzy aliasing, no semantic
    merging. Deliberately dumb: two different phrasings of 'the same'
    identity stay as two separate nodes until a human normalizes them
    later, at the corpus level."""

    def __init__(self):
        self._by_text = {}

    def get_or_create(self, raw_text: str, entry_id: str):
        if not raw_text:
            return None
        key = raw_text.strip().lower()
        if key not in self._by_text:
            cid = "ic_" + re.sub(r"[^a-z0-9]+", "_", key).strip("_")
            self._by_text[key] = {"identity_category_id": cid, "raw_text": raw_text.strip(),
                                   "first_mentioned_entry": entry_id}
        return self._by_text[key]["identity_category_id"]

    def export(self) -> list:
        return list(self._by_text.values())



## 5. HealthState tracker (not a plain registry)

This handles something a naive per-event approach would get wrong: if a
character "incurs" a fever in one entry and "dies from typhus" three
entries later, those two events are almost certainly about the *same*
underlying condition, named more vaguely at first and more specifically
once it's serious — a common diary pattern — and should create **one**
`HealthState` node with two edges into it, not two disconnected ones.

**Keyed by character alone, not character+condition name** — an earlier
version of this keyed on `(character, condition_name)` exactly, but that
fails on precisely the "fever" → "typhus" naming-drift case above, since
the two strings don't match. Keying on character alone assumes at most one
open condition per character at a time, which fits how illness is narrated
in these books; the trade-off is that if the text narrates a character
recovering from one illness and the model fails to emit a `recovers_from`
before a later unrelated `incurs`, this will wrongly merge two real,
separate illnesses into one node. Worth spot-checking in review, but a
reasonable default for a first pass. Every condition name the tracker sees
for a given open node is kept in `condition_name_history`, so the
"fever" → "typhus" progression itself is preserved rather than overwritten.


In [18]:

class HealthStateTracker:
    def __init__(self):
        self._open = {}     # character_id -> health_state_id  (one open condition per character)
        self._nodes = {}

    def get_or_create(self, character_id: str, condition_name: str, mechanism: str, entry_id: str):
        if character_id in self._open:
            hs_id = self._open[character_id]
            node = self._nodes[hs_id]
            if condition_name and condition_name not in node["condition_name_history"]:
                node["condition_name_history"].append(condition_name)
                node["condition_name"] = condition_name   # most recently stated name wins as the display name
            return hs_id
        hs_id = f"hs_{character_id}_{entry_id}_{len(self._nodes)}"
        self._nodes[hs_id] = {
            "health_state_id": hs_id, "mechanism": mechanism,
            "condition_name": condition_name,
            "condition_name_history": [condition_name] if condition_name else [],
            "first_entry": entry_id,
        }
        self._open[character_id] = hs_id
        return hs_id

    def close(self, character_id: str):
        self._open.pop(character_id, None)

    def export(self) -> list:
        return list(self._nodes.values())



## 6. Basic book context

Unchanged in purpose from v2 — still a short, factual, hand-authored
synopsis used only to disambiguate references, never as a source of facts
about specific entries. This now also gets folded into the **cached**
system prompt (§8) instead of resent per-call in the user message, since
it's fully static for a given book run.


In [27]:

BASIC_CONTEXT = {
    "SofS_entries_halved": {
        "narrator_id": "johanna_leary",
        "narrator_name": "Johanna Leary",
        "narrator_age_at_writing": 13,
        "setting_summary": (
            "Johanna Leary, 13, emigrates from Ireland during the Great Famine in 1847 with her "
            "parents, older brother Michael (14), and baby brother Patrick, bound for Canada East "
            "(Quebec) aboard a crowded emigrant ('coffin') ship. Typhus ('ship fever') spreads "
            "aboard ship and at the Grosse Île quarantine station near Quebec City; over the "
            "course of the book Johanna loses her baby brother, mother, and father to the "
            "epidemic. She and Michael are separated near the end of the book when he is wrongly "
            "told she has died and leaves to search for their uncle Liam, somewhere in Canada, "
            "leaving Johanna alone."
        ),
        "narrative_structure": "real_time",
        "instructions_note": (
            "Use this only to interpret unclear references (place names, historical events "
            "alluded to, family relationships). Do NOT use it to infer facts not stated or "
            "clearly implied in the entry text itself, and do not assume events from the synopsis "
            "have already happened at any given entry — read each entry on its own terms."
        ),
    },
}



## 7. State summary — now recency-capped

This is the main efficiency fix. In v2 the state summary grew by roughly
one entity's worth of detail every entry and was resent, in full, every
single call — on a book with *n* entries that's O(n²) tokens spent just on
restating earlier context, which is most of what the $5 / 1.66M input
tokens on the first SofS run actually went to.

Now: entities touched within the last `STATE_SUMMARY_RECENT_WINDOW`
entries get full detail (aliases, last-known-location, etc); everything
older collapses to a bare `{id, label}` pair. Combined with system-prompt
caching (§8), this should cut cost substantially without changing what
gets extracted — it only changes how much *redundant* context gets resent.


In [20]:

def build_state_summary(char_registry, loc_registry, region_registry, org_registry,
                         current_index: int, recent_window: int = STATE_SUMMARY_RECENT_WINDOW) -> dict:

    def is_recent(entry) -> bool:
        return (current_index - entry.last_touched_index) <= recent_window

    def summarize(registry, id_field, extra_fields=()):
        full, sparse = [], []
        for e in registry._by_id.values():
            eid = getattr(e, id_field)
            if is_recent(e):
                row = {"id": eid, "label": e.canonical_label, "aliases": e.aliases}
                for f in extra_fields:
                    row[f] = getattr(e, f, None)
                full.append(row)
            else:
                sparse.append({"id": eid, "label": e.canonical_label})
        return full, sparse

    chars_full, chars_sparse = summarize(char_registry, "character_id",
                                          extra_fields=("relation_to_narrator", "last_known_location"))
    locs_full, locs_sparse = summarize(loc_registry, "location_id")
    regions_full, regions_sparse = summarize(region_registry, "region_id")
    orgs_full, orgs_sparse = summarize(org_registry, "org_id")

    return {
        "known_characters_recent": chars_full, "known_characters_earlier_in_book": chars_sparse,
        "known_locations_recent": locs_full, "known_locations_earlier_in_book": locs_sparse,
        "known_regions_recent": regions_full, "known_regions_earlier_in_book": regions_sparse,
        "known_organizations_recent": orgs_full, "known_organizations_earlier_in_book": orgs_sparse,
    }



## 8. Extraction schema & prompt

The per-entry output shape, substantially restructured from v2: rather than
one flat list of triples trying to carry every kind of fact, `stays`,
`health_events`, and `incidents` are their own structured lists — matching
the reified-node shape they'll become, rather than forcing n-ary facts
(a `Stay` needs a location *and* a type *and* optional dates; a
`HealthState` needs a mechanism *and* a condition *and* an optional
perpetrator) into a binary subject/relation/object triple.

```
{
  "entry_date_raw": "...",
  "entry_date_normalized": "YYYY-MM-DD, YYYY-MM, YYYY, or null",
  "relations": [
    {"subject": "...", "subject_type": "character|organization",
     "relation": "works_for|works_with|student_of|studies_with|lives_with|
                  stops_living_with|is_related_to|social_relationship|within|located_at",
     "object": "...", "object_type": "character|organization|location|region",
     "kinship_label": "...",   # only for is_related_to, literal
     "valence": "friend|antagonist|neutral"}   # only for social_relationship
  ],
  "stays": [
    {"character": "...", "location": "...",
     "stay_type": "resident|visit|pass_through|confined|birthplace|intended",
     "mode": "...",            # optional, only if explicitly stated
     "arrival_date": "YYYY-MM-DD or null", "departure_date": "YYYY-MM-DD or null"}
  ],
  "health_events": [
    {"character": "...", "event_type": "incurs|recovers_from|dies_from",
     "mechanism": "disease|violence|accident|starvation|exposure|drowning|childbirth|other",
     "condition_name": "<specific literal term, e.g. 'typhus' — not just 'illness'>",
     "perpetrator": "... or null", "perpetrator_type": "character|organization",
     "date": "YYYY-MM-DD or null"}
  ],
  "incidents": [
    {"incident_type": "discrimination", "victim": "...",
     "perpetrator": "... or null", "perpetrator_type": "character|organization|null",
     "grounds": "... or null", "raw_text": "short literal snippet, optional"}
  ],
  "mentions": [ {"surface_form": "...", "type": "character|location|region|organization"} ],
  "identity_descriptors": [
    {"character": "...", "raw_text": "...", "attributed_by": "self|narrator|other_character|unspecified"}
  ]
}
```


In [21]:

EXTRACTION_SYSTEM_PROMPT_TEMPLATE = """You are extracting structured \
knowledge-graph data from one dated entry of a first-person \
historical-fiction diary novel (the Dear Canada series), for an academic \
project studying spatial and identity representation in Canadian \
children's historical fiction.

BOOK CONTEXT (static for this whole book — use only to disambiguate \
references, never to infer facts not stated in the entry itself):
{basic_context_json}

You will also be given, per call: a compact state summary of \
characters/locations/regions/organizations already known from earlier in \
this book, the text of the previous {context_window} entries (for \
resolving pronouns/references only — do not report facts from those \
entries as if they belong to the current one), and the current entry's text.

Return ONLY valid JSON, no preamble, no markdown fences, in this exact shape:
{{
  "entry_date_raw": "<exactly as it appears in the entry text, or null>",
  "entry_date_normalized": "<YYYY-MM-DD, or YYYY-MM / YYYY if only partially determinable, or null>",
  "relations": [
    {{"subject": "<surface form>", "subject_type": "character|organization",
     "relation": "works_for|works_with|student_of|studies_with|lives_with|stops_living_with|is_related_to|social_relationship|within|located_at",
     "object": "<surface form>", "object_type": "character|organization|location|region",
     "kinship_label": "<ONLY for is_related_to — the literal kinship word the text uses, e.g. 'uncle', 'half-brother', 'godmother'>",
     "valence": "<ONLY for social_relationship — friend|antagonist|neutral>"}}
  ],
  "stays": [
    {{"character": "<surface form>", "location": "<surface form>",
     "stay_type": "resident|visit|pass_through|confined|birthplace|intended",
     "mode": "<ONLY if explicitly stated, e.g. 'on foot', 'by cart', 'aboard the ship' — omit otherwise>",
     "arrival_date": "<YYYY-MM-DD or null>", "departure_date": "<YYYY-MM-DD or null, only if stated>"}}
  ],
  "health_events": [
    {{"character": "<surface form>", "event_type": "incurs|recovers_from|dies_from",
     "mechanism": "disease|violence|accident|starvation|exposure|drowning|childbirth|other",
     "condition_name": "<the SPECIFIC named condition if the text gives one, e.g. 'typhus', 'ship fever', 'cholera', 'consumption', 'gunshot wound' — do NOT default to generic 'illness'/'sickness' unless that is genuinely all the text says>",
     "perpetrator": "<ONLY if mechanism is violence AND a perpetrator is named, else null>",
     "perpetrator_type": "character|organization",
     "date": "<YYYY-MM-DD or null>"}}
  ],
  "incidents": [
    {{"incident_type": "discrimination", "victim": "<surface form>",
     "perpetrator": "<surface form or null>", "perpetrator_type": "character|organization|null",
     "grounds": "<literal identity-related reason if the text states one, else null>",
     "raw_text": "<short literal snippet or close paraphrase of what happened, optional>"}}
  ],
  "mentions": [ {{"surface_form": "<text>", "type": "character|location|region|organization"}} ],
  "identity_descriptors": [
    {{"character": "<surface form>", "raw_text": "<the exact word(s) the text uses>",
     "attributed_by": "self|narrator|other_character|unspecified"}}
  ]
}}

STAY GUIDANCE:
- "resident": settled/living at a place with no sense of imminent departure.
- "visit": temporarily present with clear intent to leave.
- "pass_through": en route, not stopping meaningfully.
- "confined": legally or medically restricted to a place (e.g. quarantine).
- "birthplace": stated as where the character was born.
- "intended": a stated hope, intention, or destination not yet reached (e.g. "I pray we reach Grosse Île soon").
- Only extract a stay the text actually states or clearly implies THIS entry — do not re-assert a stay just because it was true in an earlier entry; continuity is handled downstream in code, not by you.

HEALTH EVENT GUIDANCE:
- "incurs": the character falls ill / is injured. "recovers_from": explicitly recovers. "dies_from": dies.
- condition_name should be as specific as the text allows. Prefer the named illness/injury over a vague placeholder.
- perpetrator/perpetrator_type ONLY apply when mechanism is "violence" and the text names who caused it.

INCIDENT (DISCRIMINATION) GUIDANCE:
- Use only when the text describes a character being treated badly or unfairly because of who they are (ethnicity, nationality, religion, class, etc).
- victim is required; perpetrator and grounds are optional — include only what the text actually states.

IDENTITY DESCRIPTORS — read carefully:
- Extract the LITERAL words the text uses to describe a character's ethnicity, nationality, religion, class, or other identity-related characterization. Copy the phrasing, do not substitute your own category label.
  - Correct: text says "the Irish girl" -> raw_text "the Irish girl" (not "Irish").
  - Correct: text says "we are Papists" -> raw_text "Papists" (not "Catholic").
  - WRONG: generalizing or flattening any group's own specific self-description into a broader external label — this is the single biggest failure mode to avoid.
- Only extract a descriptor when the entry text itself uses identity-related language in that specific entry. Do not tag a character as an identity category just because background context (or the basic_context synopsis above) establishes it generally.

RELATION GUIDANCE:
- "within": only Location->Region or Region->Region (one direction only — do not also emit the inverse "contains").
- "located_at": Organization->Location (where an institution physically is).
- Only extract a relation the text actually states or clearly implies THIS entry. Leave things out rather than guessing.

DATE GUIDANCE:
- entry_date_normalized: read the entry's own date heading/text. Use a full YYYY-MM-DD if determinable (you may fill in an unstated year ONLY if it is genuinely unambiguous from context), otherwise a partial YYYY-MM or YYYY, otherwise null. Do not guess a full date you cannot support.
- Dates inside "stays" and "health_events" follow the same rule and should be null unless the text actually gives a date for that specific event (most will default to the entry's own date downstream in code — you do not need to fill that in yourself).

Note common historical-fiction diary conventions: narrators sometimes refer \
to family members by relationship term only ("Da", "Mother", "my brother") \
rather than by name — use the state summary's known characters to resolve \
these where possible; if genuinely new, use the surface form as written.
"""

def build_user_content(entry_row, context_entries_text, state_summary):
    payload = {
        "state_summary": state_summary,
        "previous_entries_for_reference_only": context_entries_text,
        "current_entry": {
            "entry_id": entry_row["entry_id"],
            "raw_date_column": str(entry_row["date"]),
            "entry_text": entry_row["entry_text"],
        },
    }
    return json.dumps(payload, indent=2)



## 9. Per-entry LLM call — with prompt caching

`basic_context` + the full instruction prompt are now assembled **once per
book** into a single static system string and marked
`cache_control: {"type": "ephemeral"}`. Only the state summary and the
previous-entries window (both genuinely dynamic) go in the per-call user
message. Since this is a tight sequential loop, nearly every call after the
first should hit the cache; cache hits are billed at a steep discount
relative to the same tokens processed fresh, and a cache hit also resets
the ephemeral TTL, so the whole book run should stay warm throughout.

If your SDK/account requires a beta header for prompt caching, add
`extra_headers={"anthropic-beta": "prompt-caching-2024-07-31"}` to the
`messages.create` call below — check
https://docs.claude.com for current requirements before running at scale.


In [34]:

def build_cached_system_prompt(book_code: str) -> list:
    basic_context_json = json.dumps(BASIC_CONTEXT[book_code], indent=2)
    system_text = EXTRACTION_SYSTEM_PROMPT_TEMPLATE.format(
        basic_context_json=basic_context_json, context_window=CONTEXT_WINDOW_ENTRIES,
    )
    return [{"type": "text", "text": system_text, "cache_control": {"type": "ephemeral"}}]


def call_llm_for_entry(entry_row, prior_entries_df, state_summary, cached_system):
    context_texts = [
        {"entry_id": r["entry_id"], "text": r["entry_text"]}
        for _, r in prior_entries_df.tail(CONTEXT_WINDOW_ENTRIES).iterrows()
    ]
    user_content = build_user_content(entry_row, context_texts, state_summary)

    resp = client.messages.create(
        model=MODEL_NAME,
        max_tokens=20000,
        system=cached_system,
        messages=[{"role": "user", "content": user_content}],
    )

    if resp.stop_reason == "max_tokens":
      raise RuntimeError(
          "Claude used the entire token limit before producing JSON. "
          "Increase max_tokens (try 12000) or reduce the candidate list."
      )

    # A response can contain a thinking block before its text block.
    text_blocks = [
        block.text
        for block in resp.content
        if block.type == "text"
    ]

    if not text_blocks:
        raise ValueError(
            f"No text was returned. Response blocks: "
            f"{[block.type for block in resp.content]}"
        )

    text = "\n".join(text_blocks).strip()

    text = re.sub(r"^```(json)?|```$", "", text, flags=re.MULTILINE).strip()
    try:
        return json.loads(text), resp.usage
    except json.JSONDecodeError as e:
        return {"_parse_error": str(e), "_raw_text": text}, resp.usage



## 10. Resolution: mapping model output onto registries and reified nodes

`resolve_entity` now covers character/location/region/organization only
(no more `significant_event`). Every successful resolution also `touch`es
the registry entry with the current entry index, which is what makes the
recency-capped state summary (§7) work.

`process_relation` handles the simple binary edges. `process_stay`,
`process_health_event`, and `process_incident` each materialize one node
plus its edges — this is the part that replaces the single flat
`relations_out` list from v2 with a proper per-node-type structure.


In [23]:

def resolve_entity(label, etype, char_registry, loc_registry, region_registry, org_registry, entry_id, index):
    if not label:
        return None, None
    if etype == "character":
        e = char_registry.resolve(label) or char_registry.register_or_flag(label, entry_id)
        char_registry.touch(e.character_id, index)
        return e.character_id, e.canonical_label
    if etype == "location":
        e = loc_registry.resolve(label) or loc_registry.register_or_flag(label, entry_id)
        loc_registry.touch(e.location_id, index)
        return e.location_id, e.canonical_label
    if etype == "region":
        e = region_registry.resolve(label) or region_registry.register_or_flag(label, entry_id)
        region_registry.touch(e.region_id, index)
        return e.region_id, e.canonical_label
    if etype == "organization":
        e = org_registry.resolve(label) or org_registry.register_or_flag(label, entry_id)
        org_registry.touch(e.org_id, index)
        return e.org_id, e.canonical_label
    return None, label


def make_edge(edge_type, book, entry_id, from_id, from_type, to_id, to_type, properties=None, interpretive=False):
    return {
        "edge_type": edge_type, "book": book, "entry_id": entry_id,
        "from_id": from_id, "from_type": from_type, "to_id": to_id, "to_type": to_type,
        "properties": properties or {}, "interpretive": interpretive,
    }


INTERPRETIVE_RELATIONS = {"social_relationship"}   # grieves_for removed entirely (v3); impacted_by removed (v3)


def process_relation(rel, book, entry_id, regs, index, edges_out):
    subj_id, subj_label = resolve_entity(rel.get("subject"), rel.get("subject_type"), *regs, entry_id, index)
    obj_id, obj_label = resolve_entity(rel.get("object"), rel.get("object_type"), *regs, entry_id, index)
    relation = rel.get("relation")
    props = {}
    if relation == "is_related_to":
        props["kinship_label"] = rel.get("kinship_label")
    if relation == "social_relationship":
        props["valence"] = rel.get("valence")
    edges_out.append(make_edge(relation, book, entry_id, subj_id, rel.get("subject_type"),
                                obj_id, rel.get("object_type"), props,
                                interpretive=relation in INTERPRETIVE_RELATIONS))


def process_stay(stay, book, entry_id, entry_date, regs, index, stays_out, edges_out, date_registry):
    char_id, _ = resolve_entity(stay.get("character"), "character", *regs, entry_id, index)
    loc_id, _ = resolve_entity(stay.get("location"), "location", *regs, entry_id, index)
    if char_id is None or loc_id is None:
        return
    stay_id = f"stay_{char_id}_{entry_id}_{len(stays_out)}"
    arrival = stay.get("arrival_date") or entry_date
    departure = stay.get("departure_date")
    stays_out.append({
        "stay_id": stay_id, "stay_type": stay.get("stay_type"), "mode": stay.get("mode"),
        "arrival_date": arrival, "departure_date": departure,
    })
    edges_out.append(make_edge("has_stay", book, entry_id, char_id, "character", stay_id, "stay"))
    edges_out.append(make_edge("at", book, entry_id, stay_id, "stay", loc_id, "location"))
    edges_out.append(make_edge("recorded_in", book, entry_id, stay_id, "stay", entry_id, "entry"))
    arrival_date_id = date_registry.get_or_create(arrival)
    if arrival_date_id:
        edges_out.append(make_edge("occurs_on", book, entry_id, stay_id, "stay", arrival_date_id, "date",
                                    {"date_role": "arrival"}))
    if departure:
        departure_date_id = date_registry.get_or_create(departure)
        edges_out.append(make_edge("occurs_on", book, entry_id, stay_id, "stay", departure_date_id, "date",
                                    {"date_role": "departure"}))
    # Feed the location-carry-forward state used for the eventual map.
    char_registry = regs[0]
    char_registry.update_location(char_id, stay.get("location"), entry_id)


def process_health_event(ev, book, entry_id, entry_date, regs, index, health_tracker, edges_out, date_registry):
    char_id, _ = resolve_entity(ev.get("character"), "character", *regs, entry_id, index)
    if char_id is None:
        return
    event_type = ev.get("event_type")
    condition_name = ev.get("condition_name")
    mechanism = ev.get("mechanism")
    hs_id = health_tracker.get_or_create(char_id, condition_name, mechanism, entry_id)

    edges_out.append(make_edge(event_type, book, entry_id, char_id, "character", hs_id, "health_state",
                                {"date": ev.get("date") or entry_date}))
    edges_out.append(make_edge("recorded_in", book, entry_id, hs_id, "health_state", entry_id, "entry"))
    date_id = date_registry.get_or_create(ev.get("date") or entry_date)
    if date_id:
        edges_out.append(make_edge("occurs_on", book, entry_id, hs_id, "health_state", date_id, "date"))

    if mechanism == "violence" and ev.get("perpetrator"):
        perp_id, _ = resolve_entity(ev.get("perpetrator"), ev.get("perpetrator_type") or "character",
                                     *regs, entry_id, index)
        if perp_id:
            edges_out.append(make_edge("perpetrated_by", book, entry_id, hs_id, "health_state",
                                        perp_id, ev.get("perpetrator_type") or "character"))

    if event_type in ("recovers_from", "dies_from"):
        health_tracker.close(char_id)


def process_incident(inc, book, entry_id, entry_date, regs, index, incidents_out, edges_out,
                      identity_registry, date_registry):
    victim_id, _ = resolve_entity(inc.get("victim"), "character", *regs, entry_id, index)
    if victim_id is None:
        return
    incident_id = f"incident_{victim_id}_{entry_id}_{len(incidents_out)}"
    incidents_out.append({
        "incident_id": incident_id, "incident_type": inc.get("incident_type", "discrimination"),
        "raw_text": inc.get("raw_text"),
    })
    edges_out.append(make_edge("against", book, entry_id, incident_id, "incident", victim_id, "character",
                                interpretive=True))
    edges_out.append(make_edge("recorded_in", book, entry_id, incident_id, "incident", entry_id, "entry"))
    date_id = date_registry.get_or_create(entry_date)
    if date_id:
        edges_out.append(make_edge("occurs_on", book, entry_id, incident_id, "incident", date_id, "date"))

    if inc.get("perpetrator"):
        perp_id, _ = resolve_entity(inc.get("perpetrator"), inc.get("perpetrator_type") or "character",
                                     *regs, entry_id, index)
        if perp_id:
            edges_out.append(make_edge("perpetrated_by", book, entry_id, incident_id, "incident",
                                        perp_id, inc.get("perpetrator_type") or "character", interpretive=True))

    if inc.get("grounds"):
        ic_id = identity_registry.get_or_create(inc["grounds"], entry_id)
        edges_out.append(make_edge("on_grounds_of", book, entry_id, incident_id, "incident",
                                    ic_id, "identity_category", interpretive=True))


## 11. Process a book

In [35]:

def process_book(book_code: str, df: pd.DataFrame, reviewed: dict):
    char_registry = CharacterRegistry()
    loc_registry = LocationRegistry()
    region_registry = RegionRegistry()
    org_registry = OrganizationRegistry()
    date_registry = DateRegistry()
    identity_registry = IdentityCategoryRegistry()
    health_tracker = HealthStateTracker()

    char_registry.seed_from_reviewed(reviewed.get("characters", []))
    loc_registry.seed_from_reviewed(reviewed.get("locations", []))
    region_registry.seed_from_reviewed(reviewed.get("regions", []))
    org_registry.seed_from_reviewed(reviewed.get("organizations", []))

    basic_context = BASIC_CONTEXT[book_code]
    char_registry.seed_narrator(basic_context["narrator_id"], basic_context["narrator_name"])
    cached_system = build_cached_system_prompt(book_code)

    regs = (char_registry, loc_registry, region_registry, org_registry)

    entries_out, edges_out, stays_out, incidents_out, mentions_out = [], [], [], [], []
    parse_errors = []
    total_usage = {"input_tokens": 0, "output_tokens": 0, "cache_read_input_tokens": 0, "cache_creation_input_tokens": 0}

    with tqdm(
        df.iterrows(),
        total=len(df),
        desc=f"Processing {book_code}",
        unit="entry",
    ) as progress:
        for i, row in progress:
            prior_entries_df = df.iloc[:i]
            state_summary = build_state_summary(char_registry, loc_registry, region_registry, org_registry, current_index=i)
            result, usage = call_llm_for_entry(row, prior_entries_df, state_summary, cached_system)

            for k in total_usage:
                total_usage[k] += getattr(usage, k, 0) or 0

            if "_parse_error" in result:
                parse_errors.append({"entry_id": row["entry_id"], **result})
                continue

            entry_date = result.get("entry_date_normalized")
            entries_out.append({
                "entry_id": row["entry_id"], "book": book_code, "chapter_id": row["chapter_id"],
                "entry_date_raw": result.get("entry_date_raw"), "entry_date": entry_date,
            })
            date_id = date_registry.get_or_create(entry_date)
            if date_id:
                edges_out.append(make_edge("occurs_on", book_code, row["entry_id"], row["entry_id"], "entry",
                                            date_id, "date"))

            for rel in result.get("relations", []):
                process_relation(rel, book_code, row["entry_id"], regs, i, edges_out)

            for stay in result.get("stays", []):
                process_stay(stay, book_code, row["entry_id"], entry_date, regs, i, stays_out, edges_out, date_registry)

            for ev in result.get("health_events", []):
                process_health_event(ev, book_code, row["entry_id"], entry_date, regs, i, health_tracker,
                                    edges_out, date_registry)

            for inc in result.get("incidents", []):
                process_incident(inc, book_code, row["entry_id"], entry_date, regs, i, incidents_out, edges_out,
                                identity_registry, date_registry)

            for m in result.get("mentions", []):
                eid, elabel = resolve_entity(m.get("surface_form"), m.get("type"), *regs, row["entry_id"], i)
                mentions_out.append({"entry_id": row["entry_id"], "entity_id": eid, "entity_label": elabel,
                                    "entity_type": m.get("type")})

            for d in result.get("identity_descriptors", []):
                char_id, char_label = resolve_entity(d.get("character"), "character", *regs, row["entry_id"], i)
                ic_id = identity_registry.get_or_create(d.get("raw_text"), row["entry_id"])
                if char_id and ic_id:
                    edges_out.append(make_edge("has_identity_descriptor", book_code, row["entry_id"],
                                                char_id, "character", ic_id, "identity_category",
                                                {"attributed_by": d.get("attributed_by", "unspecified")}))

            progress.set_postfix(
                errors=len(parse_errors),
                in_tokens=f"{total_usage['input_tokens']:,}",
                out_tokens=f"{total_usage['output_tokens']:,}",
            )

    return {
        "entries": entries_out, "edges": edges_out, "stays": stays_out,
        "incidents": incidents_out, "mentions": mentions_out, "parse_errors": parse_errors,
        "usage": total_usage,
        "registries": {
            "characters": char_registry, "locations": loc_registry, "regions": region_registry,
            "organizations": org_registry, "dates": date_registry, "identity_categories": identity_registry,
            "health_states": health_tracker,
        },
    }



## 12. Export

One file per node type, one flat `edges.json`. The map-building step
(building per-character location-over-time from `stay`/`at`/`occurs_on`
edges) is left for a separate later pass once you've seen this output —
it's a pure read/transform over `edges.json`, not something that needs to
happen inside extraction.


In [25]:

def export_book(book_code: str, result: dict):
    with open(OUT_DIR / f"{book_code}_nodes_entries.json", "w") as f:
        json.dump(result["entries"], f, indent=2)
    with open(OUT_DIR / f"{book_code}_nodes_stays.json", "w") as f:
        json.dump(result["stays"], f, indent=2)
    with open(OUT_DIR / f"{book_code}_nodes_incidents.json", "w") as f:
        json.dump(result["incidents"], f, indent=2)
    with open(OUT_DIR / f"{book_code}_edges.json", "w") as f:
        json.dump(result["edges"], f, indent=2)
    with open(OUT_DIR / f"{book_code}_mentions.json", "w") as f:
        json.dump(result["mentions"], f, indent=2)

    regs = result["registries"]
    with open(REGISTRY_DIR / f"{book_code}_nodes_characters.json", "w") as f:
        json.dump(regs["characters"].export(), f, indent=2)
    with open(REGISTRY_DIR / f"{book_code}_nodes_locations.json", "w") as f:
        json.dump(regs["locations"].export(), f, indent=2)
    with open(REGISTRY_DIR / f"{book_code}_nodes_regions.json", "w") as f:
        json.dump(regs["regions"].export(), f, indent=2)
    with open(REGISTRY_DIR / f"{book_code}_nodes_organizations.json", "w") as f:
        json.dump(regs["organizations"].export(), f, indent=2)
    with open(REGISTRY_DIR / f"{book_code}_nodes_dates.json", "w") as f:
        json.dump(regs["dates"].export(), f, indent=2)
    with open(REGISTRY_DIR / f"{book_code}_nodes_identity_categories.json", "w") as f:
        json.dump(regs["identity_categories"].export(), f, indent=2)
    with open(REGISTRY_DIR / f"{book_code}_nodes_health_states.json", "w") as f:
        json.dump(regs["health_states"].export(), f, indent=2)

    with open(REVIEW_DIR / f"{book_code}_pending_characters.json", "w") as f:
        json.dump(regs["characters"].pending_review, f, indent=2)
    with open(REVIEW_DIR / f"{book_code}_pending_locations.json", "w") as f:
        json.dump(regs["locations"].pending_review, f, indent=2)
    with open(REVIEW_DIR / f"{book_code}_pending_regions.json", "w") as f:
        json.dump(regs["regions"].pending_review, f, indent=2)
    with open(REVIEW_DIR / f"{book_code}_pending_organizations.json", "w") as f:
        json.dump(regs["organizations"].pending_review, f, indent=2)
    if result["parse_errors"]:
        with open(REVIEW_DIR / f"{book_code}_parse_errors.json", "w") as f:
            json.dump(result["parse_errors"], f, indent=2)

    n_interpretive = sum(1 for e in result["edges"] if e["interpretive"])
    u = result["usage"]
    print(f"[{book_code}] {len(result['entries'])} entries, {len(result['edges'])} edges "
          f"({n_interpretive} interpretive), {len(result['stays'])} stays, "
          f"{len(result['incidents'])} incidents, {len(regs['health_states'].export())} health states.")
    print(f"  Tokens — input: {u['input_tokens']}, output: {u['output_tokens']}, "
          f"cache read: {u['cache_read_input_tokens']}, cache write: {u['cache_creation_input_tokens']}")
    print(f"  Flagged for review: {len(regs['characters'].pending_review)} characters, "
          f"{len(regs['locations'].pending_review)} locations, {len(regs['regions'].pending_review)} regions, "
          f"{len(regs['organizations'].pending_review)} organizations.")
    if result["parse_errors"]:
        print(f"  {len(result['parse_errors'])} entries failed to parse — see {book_code}_parse_errors.json")



## 13. Running this on *A Sea of Sorrows* — setup details

### Do you need to delete v2's files first?

**No deletion required, but don't try to reuse v2's output.** This
notebook writes to `./output/v3/...` (a new, separate directory tree), so
it will not overwrite or collide with whatever is sitting in `./output/`
from v2. You can leave the old files in place, move them aside, or delete
them — it makes no functional difference, since v3 doesn't read anything
v2 produced.

**What you *do* need to redo:** the candidate-review file
(`output/candidate_lists/SofS_reviewed.json`). Its schema changed — it now
needs separate `locations`/`regions` lists (v2 had one combined
`locations` list) plus a `historicity_hint` per character and
`org_type_hint` per organization. The underlying entities haven't changed,
so re-running `suggest_canonical_entities` (§3) fresh is almost certainly
less work than hand-editing the old file. The raw NER candidate list
(`SofS_candidates.json`) and the raw CSV (`./data/SofS.csv`) are unaffected
and don't need to be regenerated.

### Full setup checklist

1. **Directory layout**, relative to wherever you run this notebook:
   ```
   ./data/SofS.csv                        (your input — unchanged from v2)
   ./output/candidate_lists/               (shared across versions)
   ./output/v3/                            (created automatically)
     SofS_nodes_entries.json
     SofS_nodes_stays.json
     SofS_nodes_incidents.json
     SofS_edges.json
     SofS_mentions.json
     registries/
       SofS_nodes_characters.json
       SofS_nodes_locations.json
       SofS_nodes_regions.json
       SofS_nodes_organizations.json
       SofS_nodes_dates.json
       SofS_nodes_identity_categories.json
       SofS_nodes_health_states.json
     for_review/
       SofS_pending_characters.json  (etc.)
   ```
2. **Environment**: `ANTHROPIC_API_KEY` set; `anthropic`, `pandas`, `spacy`
   installed; `en_core_web_sm` or `en_core_web_trf` downloaded.
3. **Run order**: §1 load CSV → §2 NER pass → §3 review (produce
   `SofS_reviewed.json`) → §4–§5 registries are defined (no action needed,
   just run the cells) → `reviewed = load_reviewed_entities("SofS")` →
   `result = process_book("SofS", book_df, reviewed)` → `export_book("SofS", result)`.
4. **What to check first this run**: the printed token/cache-read counts
   (should show a large `cache_read_input_tokens` figure from the second
   entry onward — if it's near zero, caching isn't engaging and is worth
   debugging before running the full corpus); `for_review/*_pending_*.json`
   for anything auto-registered that looks wrong; a few `health_events`
   entries by eye once the family starts falling ill, to check `condition_name`
   is coming through as "typhus"/"ship fever" rather than generic "illness",
   and that a single character's incur→die sequence produced one
   `HealthState` node (check `SofS_nodes_health_states.json`) rather than two.


In [36]:
result = process_book("SofS_entries_halved", book_df, reviewed)
export_book("SofS_entries_halved", result)

Processing SofS_entries_halved:   0%|          | 0/137 [00:00<?, ?entry/s]

[SofS_entries_halved] 137 entries, 1450 edges (116 interpretive), 181 stays, 22 incidents, 45 health states.
  Tokens — input: 1258037, output: 257331, cache read: 372600, cache write: 5520
  Flagged for review: 142 characters, 56 locations, 6 regions, 16 organizations.
